In [2]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

# Loading data

In [3]:
learn_data = pd.read_csv("preprocess_train_v3.csv", header = None)
learn_data.columns = ['Age', 'ALB', 'AR', 'IBRatio', 'AST_ALT_Ratio', 'LogIB', 'LogAlkphos', 'LogSgpt', 'LogSgot', 'Female', 'Target']
learn_data["Female"] = learn_data["Female"].astype("category")
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,ALB,AR,IBRatio,AST_ALT_Ratio,LogIB,LogAlkphos,LogSgpt,LogSgot,Female,Target
0,48,2.4,0.52,0.488889,5.692308,7.884574e-01,5.641907,2.564949,4.304065,0,0
1,39,4.3,1.38,0.526316,1.476190,-1.110223e-16,5.192957,3.737670,4.127134,0,0
2,23,3.1,1.00,0.700000,1.951220,-3.566749e-01,5.356586,3.713572,4.382027,0,0
3,42,3.2,1.06,0.714286,2.314286,-6.931472e-01,5.023881,3.555348,4.394449,1,0
4,54,3.4,0.80,0.495575,1.233333,2.415914e+00,6.324359,3.401197,3.610918,1,0


In [4]:
learn_data.isna().value_counts()

Age    ALB    AR     IBRatio  AST_ALT_Ratio  LogIB  LogAlkphos  LogSgpt  LogSgot  Female  Target
False  False  False  False    False          False  False       False    False    False   False     451
Name: count, dtype: int64

In [5]:
X = learn_data.drop(columns = ["Target"])
Xnum = learn_data.drop(columns = ["Female", "Target"])
y = learn_data["Target"]

# Metrics

In [6]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

# The classifiers, by themselves

## Logistic Regression

In [16]:
logreg_pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression(class_weight = "balanced"))])

n = 100
m = 10
Cs = np.logspace(start = -4, stop = 2, num = n)

logreg_search = GridSearchCV(estimator = logreg_pipeline,
                             param_grid = {'logreg__C' : Cs},
                             scoring = 'f1_macro',
                             cv = 5)
logreg_search.fit(Xnum, y)
logreg_search.best_params_

{'logreg__C': 0.10722672220103231}

In [17]:
logreg_search.best_score_

0.6561321598937985

In [19]:
logreg_C = logreg_search.best_params_["logreg__C"]

logreg_model_best = LogisticRegression(C = logreg_C,
                                       class_weight = "balanced")
logreg_pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', logreg_model_best)])

cross_val_results = pd.DataFrame(cross_validate(logreg_pipeline, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["LogReg-Best", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.656132,0.715346,0.67638,0.66967
QDA,0.63314,0.701341,0.666148,0.643004


## Sigmoid kernel SVC

In [20]:
sigsvc = SVC(kernel = "sigmoid", class_weight = "balanced")
sigsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", sigsvc)])

n = 50
m = 50
Cs = np.logspace(start = -1, stop = 2, num = n)
gammas = np.logspace(start = -2, stop = 2, num = m) / X.shape[0]

sigsvc_search = GridSearchCV(estimator = sigsvc_pipeline,
                             param_grid = {'svc__C' : Cs,
                                           'svc__gamma' : gammas},
                             scoring = 'f1_macro',
                             cv = 5)
sigsvc_search.fit(Xnum, y)
sigsvc_search.best_params_

{'svc__C': 75.43120063354614, 'svc__gamma': 0.015957553725080963}

In [21]:
sigsvc_search.best_score_

0.6453528289148487

In [48]:
sigsvc_C = sigsvc_search.best_params_['svc__C']
sigsvc_gamma = sigsvc_search.best_params_['svc__gamma']
sigsvc_best = SVC(kernel = "sigmoid",
                  C = sigsvc_C,
                  gamma = sigsvc_gamma,
                  class_weight = "balanced",
                  probability = True)
sigsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", sigsvc_best)])

cross_val_results = pd.DataFrame(cross_validate(sigsvc_pipeline, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Sigmoid SVC", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Random Forest,0.659621,0.712861,0.674698,0.676313
LogReg-Best,0.656132,0.715346,0.67638,0.66967
Sigmoid SVC,0.645353,0.699913,0.663325,0.66083
QDA,0.63314,0.701341,0.666148,0.643004


## Polynomial kernel SVC

In [66]:
polysvc = SVC(kernel = "poly", class_weight = "balanced")
polysvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", polysvc)])

n = 25
m = 25
Cs = np.logspace(start = -1, stop = 2, num = n)
gammas = np.logspace(start = -2, stop = 2, num = m) / X.shape[0]
degrees = [2, 3]

polysvc_search = GridSearchCV(estimator = polysvc_pipeline,
                              param_grid = {'svc__C' : Cs,
                                            'svc__gamma' : gammas,
                                            'svc__degree' : degrees},
                              scoring = 'f1_macro',
                              cv = 5)
polysvc_search.fit(Xnum, y)
polysvc_search.best_params_

{'svc__C': 1.0, 'svc__degree': 3, 'svc__gamma': 0.22172949002217296}

In [67]:
polysvc_search.best_score_

0.6438724887264813

In [74]:
polysvc_C = polysvc_search.best_params_['svc__C']
polysvc_gamma = polysvc_search.best_params_['svc__gamma']
polysvc_degree = polysvc_search.best_params_['svc__degree']
polysvc_best = SVC(kernel = "poly",
                   C = polysvc_C,
                   gamma = polysvc_gamma,
                   degree = polysvc_degree,
                   class_weight = "balanced",
                   probability = True)
polysvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", polysvc_best)])

cross_val_results = pd.DataFrame(cross_validate(polysvc_pipeline, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Polynomial SVC", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Voting,0.666065,0.676788,0.662915,0.714066
Random Forest,0.659621,0.712861,0.674698,0.676313
LogReg-Best,0.656132,0.715346,0.67638,0.66967
Sigmoid SVC,0.645353,0.699913,0.663325,0.66083
Polynomial SVC,0.643872,0.69499,0.66014,0.660733
QDA,0.63314,0.701341,0.666148,0.643004


## Random Forest

In [32]:
X_noratio = Xnum.drop(columns = "AST_ALT_Ratio")

In [159]:
from imblearn.pipeline import Pipeline as PipelineIMB
from imblearn.under_sampling import RandomUnderSampler

rf_pipeline = PipelineIMB([('scaler', StandardScaler()),
                           ('rf', RandomForestClassifier(class_weight = "balanced"))])

criteria = ["gini", "entropy", "log_loss"]
max_features = ["sqrt", "log2", None]
depths = [2, 4, 6, 8, 10]
ns = [100]
min_samples = [5, 10, 20, 30, 40, 50]
oob_score = [True]

rf_search = GridSearchCV(estimator = rf_pipeline,
                         param_grid = {'rf__criterion' : criteria,
                                      'rf__max_features' : max_features,
                                      'rf__max_depth' : depths,
                                      'rf__min_samples_split' : min_samples,
                                      'rf__n_estimators' : ns,
                                      'rf__oob_score' : oob_score},
                         scoring = 'f1_macro',
                         cv = 5)
rf_search.fit(Xnum, y)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('rf',
                                        RandomForestClassifier(class_weight='balanced'))]),
             param_grid={'rf__criterion': ['gini', 'entropy', 'log_loss'],
                         'rf__max_depth': [2, 4, 6, 8, 10],
                         'rf__max_features': ['sqrt', 'log2', None],
                         'rf__min_samples_split': [5, 10, 20, 30, 40, 50],
                         'rf__n_estimators': [100], 'rf__oob_score': [True]},
             scoring='f1_macro')

In [161]:
rf_search.best_score_

0.6741187981932303

In [162]:
rf_search.best_params_

{'rf__criterion': 'gini',
 'rf__max_depth': 8,
 'rf__max_features': 'sqrt',
 'rf__min_samples_split': 30,
 'rf__n_estimators': 100,
 'rf__oob_score': True}

In [182]:
rf_criterion = rf_search.best_params_['rf__criterion']
rf_max_depth = rf_search.best_params_['rf__max_depth']
rf_max_features = rf_search.best_params_['rf__max_features']
rf_min_samples_split = rf_search.best_params_['rf__min_samples_split']
rf_n_estimators = rf_search.best_params_['rf__n_estimators']
rf_best = RandomForestClassifier(criterion = rf_criterion,
                                max_depth = rf_max_depth,
                                max_features = rf_max_features,
                                min_samples_split = rf_min_samples_split,
                                n_estimators = rf_n_estimators,
                                class_weight = "balanced",
                                random_state = 123)
rf_pipeline = PipelineIMB([("scaler", StandardScaler()),
                           ('rf', rf_best)])

cross_val_results = pd.DataFrame(cross_validate(rf_pipeline, Xnum, y, cv = 8, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Random Forest", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Voting,0.676116,0.719053,0.681253,0.698462
LogReg-Best,0.656132,0.715346,0.67638,0.66967
Random Forest,0.649999,0.672092,0.652732,0.687343
Sigmoid SVC,0.645353,0.699913,0.663325,0.66083
Polynomial SVC,0.643872,0.69499,0.66014,0.660733
QDA,0.63314,0.701341,0.666148,0.643004


# Voting classifiers

The classifiers, by themselves, have similar performance. However, when predicting the training data they show to have different opinions. A consensus should be established.

In [183]:
qda_pipeline.fit(Xnum, y)
qda_labels = qda_pipeline.predict(Xnum)
logreg_pipeline.fit(Xnum, y)
logreg_labels = logreg_pipeline.predict(Xnum)
sigsvc_pipeline.fit(Xnum, y)
sigsvc_labels = sigsvc_pipeline.predict(Xnum)
polysvc_pipeline.fit(Xnum, y)
polysvc_labels = polysvc_pipeline.predict(Xnum)
rf_pipeline.fit(Xnum, y)
rf_labels = rf_pipeline.predict(Xnum)

In [184]:
pd.Series(np.logical_and(logreg_labels == sigsvc_labels,
                         logreg_labels == rf_labels,
                         logreg_labels == polysvc_labels)).value_counts()

True     359
False     92
Name: count, dtype: int64

Let's make them vote, then.

In [185]:
estimators = [("logreg", logreg_pipeline),
              ("sigsvc", sigsvc_pipeline),
              ("polysvc", polysvc_pipeline),
              ("rf", rf_pipeline)]
votingclass = VotingClassifier(estimators = estimators)

n = 10
weights = [(p1 / n, p2 / n, p3 / n, 1 - (p1 + p2 + p3) / n)
           for p1 in range(0, n + 1)
           for p2 in range(0, n + 1 - p1)
           for p3 in range(0, n + 1 - p1 - p2)]
modes = ["soft", "hard"]

vote_search = GridSearchCV(estimator = votingclass,
                           param_grid = {"weights" : weights,
                                         "voting" : modes},
                           scoring = "f1_macro",
                           cv = 5)
vote_search.fit(Xnum, y)
vote_search.best_params_

{'voting': 'hard', 'weights': (0.0, 0.5, 0.5, 0.0)}

In [186]:
vote_search.best_score_

0.6819568041125912

In [206]:
selected_estimators = [("sigsvc", sigsvc_pipeline),
                       ("polysvc", polysvc_pipeline)]
vote_mode = vote_search.best_params_['voting']
vote_best = VotingClassifier(estimators = selected_estimators,
                             voting = vote_mode)

cross_val_results = pd.DataFrame(cross_validate(vote_best, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Voting", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Voting,0.681957,0.703721,0.677278,0.716288
LogReg-Best,0.656132,0.715346,0.67638,0.66967
Random Forest,0.649999,0.672092,0.652732,0.687343
Sigmoid SVC,0.645353,0.699913,0.663325,0.66083
Polynomial SVC,0.643872,0.69499,0.66014,0.660733
QDA,0.63314,0.701341,0.666148,0.643004


### Making predictions

In [201]:
test_data = pd.read_csv("preprocess_test_v3.csv", header = None)
test_data.columns = ['Age', 'ALB', 'AR', 'IBRatio', 'AST_ALT_Ratio', 'LogIB', 'LogAlkphos', 'LogSgpt', 'LogSgot', 'Female']
test_data_num = test_data.drop(columns = "Female")
test_data["Female"] = learn_data["Female"].astype("category")
test_data.head()

,Age,ALB,AR,IBRatio,AST_ALT_Ratio,LogIB,LogAlkphos,LogSgpt,LogSgot,Female
0,11,4.2,1.40,0.857143,1.115385,-0.510826,6.383507,3.258097,3.367296,0
1,62,4.0,0.80,0.500000,2.246377,-0.105361,5.411646,4.234107,5.043425,0
2,60,4.2,1.10,0.714286,0.437500,-0.693147,5.159055,3.465736,2.639057,0
3,60,3.2,0.78,0.508772,2.063107,1.064711,5.365976,6.021023,6.745236,1
4,48,2.7,0.90,0.777778,2.250000,-0.356675,5.164786,3.178054,3.988984,1


In [203]:
rf_pipeline.fit(Xnum, y)

labels_rf = pd.DataFrame(columns = ['ID', 'Label'])
labels_rf['Label'] = pd.DataFrame(rf_pipeline.predict(test_data_num))
labels_rf['ID'] = labels_vote.index + 1
labels_rf.to_csv('new_predictions/rf_best_fs.csv', index = False)
labels_rf

,ID,Label
0,1,0
1,2,0
2,3,1
3,4,0
4,5,0
...,...,...
111,112,0
112,113,0
113,114,1
114,115,0


In [204]:
vote_best.fit(Xnum, y)

labels_vote = pd.DataFrame(columns = ['ID', 'Label'])
labels_vote['Label'] = pd.DataFrame(vote_best.predict(test_data_num))
labels_vote['ID'] = labels_vote.index + 1
labels_vote.to_csv('new_predictions/vote_best_fs.csv', index = False)
labels_vote

,ID,Label
0,1,1
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,1
113,114,1
114,115,1


# Bagging Classifier

In [ ]:
from sklearn.ensemble import BaggingClassifier

estimators = [("polysvc", polysvc_best), ("sigsvc", sigsvc_best)]
votingclass = VotingClassifier(estimators = estimators, voting = "soft")
bagger = BaggingClassifier(estimator = votingclass, max_samples = 0.7, n_estimators = 100)
bag_pipeline = Pipeline([("scaler", StandardScaler()),
                         ("bagger", bagger)])

cross_val_results = pd.DataFrame(cross_validate(bagger, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Bagging del Voting", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

# AdaBoost Classifier

In [209]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, cross_validate

estimators = [("polysvc", polysvc_best), ("sigsvc", sigsvc_best)]
votingclass = VotingClassifier(estimators = estimators, voting = "soft")
adaboost = AdaBoostClassifier(estimator = votingclass, n_estimators = 25)
boost_pipeline = Pipeline([("scaler", StandardScaler()),
                           ("boost", adaboost)])

cross_val_results = pd.DataFrame(cross_validate(boost_pipeline, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["AdaBoost", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Voting,0.681957,0.703721,0.677278,0.716288
LogReg-Best,0.656132,0.715346,0.67638,0.66967
Random Forest,0.649999,0.672092,0.652732,0.687343
Sigmoid SVC,0.645353,0.699913,0.663325,0.66083
Polynomial SVC,0.643872,0.69499,0.66014,0.660733
QDA,0.63314,0.701341,0.666148,0.643004
AdaBoost,0.480825,0.521702,0.584353,0.705104


In [216]:
n = 20
m = 10
learning_rates = np.logspace(start = -2, stop = 1, num = n)
n_estimators = [5, 10, 20, 40, 70, 100]

ada_search = GridSearchCV(estimator = boost_pipeline,
                          param_grid = {"boost__n_estimators" : n_estimators,
                                        "boost__learning_rate" : learning_rates},
                          cv = 5,
                          scoring = "f1_macro")
ada_search.fit(X, y)
ada_search.best_params_

{'boost__learning_rate': 0.01438449888287663, 'boost__n_estimators': 20}

In [217]:
ada_search.best_score_

0.5009633608893018

In [218]:
adaboost = AdaBoostClassifier(estimator = votingclass, learning_rate = 0.5, n_estimators = 100, random_state = 42)
boost_pipeline = Pipeline([("scaler", StandardScaler()),
                           ("boost", adaboost)])

cross_val_results = pd.DataFrame(cross_validate(boost_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["AdaBoost", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Voting,0.681957,0.703721,0.677278,0.716288
LogReg-Best,0.656132,0.715346,0.67638,0.66967
Random Forest,0.649999,0.672092,0.652732,0.687343
Sigmoid SVC,0.645353,0.699913,0.663325,0.66083
Polynomial SVC,0.643872,0.69499,0.66014,0.660733
QDA,0.63314,0.701341,0.666148,0.643004
AdaBoost,0.492889,0.519274,0.560716,0.691917
Adaboost,0.492889,0.519274,0.560716,0.691917
